In [61]:
import pandas as pd
import numpy as np

In [62]:
df = pd.read_csv("../Data/spam.csv" , encoding='latin-1')

In [63]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [64]:
df = df[['v1' , 'v2']]
df.columns = ['label', 'message' ]
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [65]:
df["label"] = df["label"].map({
    "ham": 0,
    "spam": 1
})

In [66]:
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["message"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [67]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

In [68]:
print(X_train.shape)

(4457, 7701)


In [69]:
class MultinomialNB:
    
    def __init__(self , alpha = 1):
        self.alpha = alpha
        self.classes = None
        self.prior = None
        self.features_probab = None
        
    def fit(self , X ,y ):
        X = X.toarray()
        self.classes = np.unique(y)
        number_classes = len(self.classes)
        number_features = X.shape[1]
        
        self.prior  = np.zeros(number_classes)
        for i , c  in enumerate(self.classes):
            X_class = X[y==c]
            self.prior[i] = np.log( len(X_class)/len(X))
            
            
        self.features_probab = np.zeros((number_classes ,number_features))
        
        for i , c in enumerate(self.classes):
            X_Class = X[y==c]
            word_counts = np.sum(X_Class , axis = 0)
            total_words = np.sum(word_counts)
            probability = (word_counts + self.alpha )/(total_words + self.alpha*number_features)
            self.features_probab[i] = np.log(probability)
            
        return self
    
    
    def predict(self, X):
        prediction = []
        
        for x in X:
            class_score = []
            for i, c in enumerate(self.classes):
                score = self.prior[i]
                score += np.sum(x * self.features_probab[i])
                class_score.append(score)

            best_class = self.classes[np.argmax(class_score)]
            prediction.append(best_class)

        return np.array(prediction)
            

In [70]:
my_nb = MultinomialNB(alpha=1.0)

my_nb.fit(X_train,y_train)

In [71]:
y_pred_scratch = my_nb.predict(X_test)

In [72]:
from sklearn.naive_bayes import MultinomialNB

sk_nb = MultinomialNB(alpha=1.0)
sk_nb.fit(X_train, y_train)

y_pred_sklearn = sk_nb.predict(X_test)

In [73]:
from sklearn.metrics import confusion_matrix

cm1 = confusion_matrix(y_test, y_pred_scratch)
print(cm1)

cm2 = confusion_matrix(y_test, y_pred_sklearn)
print(cm2)

[[961   5]
 [ 13 136]]
[[961   5]
 [ 13 136]]


In [ ]:
from sklearn.metrics import accuracy_score

scratch_accuracy = accuracy_score(
    y_test,
    y_pred_scratch
)

sklearn_accuracy = accuracy_score(
    y_test,
    y_pred_sklearn
)

print("Scratch :", scratch_accuracy)
print("Sklearn :", sklearn_accuracy)